# XAI-DIF — Proof of Concept
### Explainable AI-Driven Decision Intelligence Framework for Software Project Management

This notebook is a **runnable proof of concept** for the three-layer framework in the paper:

1. **Predictive Layer** — ensemble models (Random Forest, XGBoost) predict whether a project is *at risk*.
2. **Explainability Layer** — SHAP + LIME explain *why* each project is flagged.
3. **Decision Intelligence Layer** — a PMBOK-grounded rule engine turns the explanation into **prioritized, actionable recommendations**.

It also runs the three evaluations tied to your hypotheses: **H1** (explanations), **H2** (decision quality), **H3** (multi-source > single-source).

> **The data workaround.** Real enterprise Jira/GitHub data is confidential and hard to label. So this PoC *generates a realistic synthetic multi-source dataset* whose risk labels follow an interpretable rule. That is enough to demonstrate the full pipeline end-to-end. The last section shows exactly where to plug in real Jira/GitHub exports later — the rest of the code does not change.

> **Optional libraries.** The notebook prefers `xgboost`, `shap`, and `lime`, but **falls back gracefully** (GradientBoosting, permutation importance, and a z-score attribution) if they are missing — so it always runs. Install them to unlock the full path:
> ```
> pip install xgboost shap lime
> ```


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

RNG_SEED = 42
np.random.seed(RNG_SEED)

# --- optional libraries: prefer them, fall back if absent ---
try:
    import xgboost as xgb; HAVE_XGB = True
except Exception: HAVE_XGB = False
try:
    import shap; HAVE_SHAP = True
except Exception: HAVE_SHAP = False
try:
    import lime, lime.lime_tabular; HAVE_LIME = True
except Exception: HAVE_LIME = False

print(f"xgboost={HAVE_XGB}  shap={HAVE_SHAP}  lime={HAVE_LIME}")
print("Install 'xgboost shap lime' to unlock the full explainability path.")

## 1. Multi-source project data (the workaround)

Each project row combines three data sources, mirroring the paper's multi-source design:

| Source | Example features |
|---|---|
| **Task board (Jira / Azure DevOps)** | open blockers, scope-change %, avg cycle time, overdue ratio, backlog growth |
| **Version control (GitHub)** | commit frequency, PR review latency, code churn, contributors, failed-build ratio |
| **Communication** | sentiment, response latency, meeting load |

The `at_risk` label is generated from an interpretable latent function of these features (plus noise), so the models have a *real* signal to learn and the explanations are meaningful.

In [ ]:
FEATURE_GROUPS = {
    "task_board": ["open_blockers","scope_change_pct","avg_cycle_time","overdue_ratio","backlog_growth"],
    "vcs":        ["commit_freq","pr_latency","code_churn","contributors","failed_build_ratio"],
    "comms":      ["sentiment","comm_latency","meeting_load"],
    "context":    ["team_size","req_volatility"],
}
ALL_FEATURES = [f for g in FEATURE_GROUPS.values() for f in g]

def generate_project_data(n=1600, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({
        "open_blockers":     rng.poisson(2.2, n).astype(float),
        "scope_change_pct":  np.clip(rng.normal(12, 8, n), 0, 60),
        "avg_cycle_time":    np.clip(rng.normal(4.5, 2.0, n), 0.5, 15),
        "overdue_ratio":     np.clip(rng.beta(2, 6, n), 0, 1),
        "backlog_growth":    np.clip(rng.normal(5, 6, n), -10, 30),
        "commit_freq":       np.clip(rng.normal(28, 12, n), 1, 80),
        "pr_latency":        np.clip(rng.normal(20, 15, n), 1, 120),
        "code_churn":        np.clip(rng.beta(2, 5, n), 0, 1),
        "contributors":      rng.integers(2, 15, n).astype(float),
        "failed_build_ratio":np.clip(rng.beta(2, 8, n), 0, 1),
        "sentiment":         np.clip(rng.normal(0.15, 0.35, n), -1, 1),
        "comm_latency":      np.clip(rng.normal(6, 4, n), 0.2, 40),
        "meeting_load":      np.clip(rng.normal(6, 3, n), 0, 25),
        "team_size":         rng.integers(3, 20, n).astype(float),
        "req_volatility":    np.clip(rng.normal(0.3, 0.2, n), 0, 1),
    })
    # interpretable latent risk score -> probability -> noisy label
    z = (0.42*d.open_blockers + 0.03*d.scope_change_pct + 0.22*d.avg_cycle_time
         + 2.4*d.overdue_ratio + 0.05*d.backlog_growth
         - 0.03*d.commit_freq + 0.02*d.pr_latency + 1.4*d.code_churn
         + 2.0*d.failed_build_ratio - 1.6*d.sentiment + 0.06*d.comm_latency
         + 1.2*d.req_volatility)
    z = (z - z.mean()) / z.std()
    d["at_risk"] = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
    return d

df = generate_project_data()
print("rows:", len(df), "| at-risk rate:", round(df.at_risk.mean(), 3))
df.head()

## 2. Predictive Layer

Train Random Forest and XGBoost (or a GradientBoosting fallback), and report Precision, Recall, F1, and ROC-AUC.

In [ ]:
X, y = df[ALL_FEATURES], df["at_risk"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RNG_SEED)

def evaluate(model, Xe, ye, name):
    pred, proba = model.predict(Xe), model.predict_proba(Xe)[:,1]
    return dict(model=name,
                precision=round(precision_score(ye,pred),3),
                recall=round(recall_score(ye,pred),3),
                f1=round(f1_score(ye,pred),3),
                roc_auc=round(roc_auc_score(ye,proba),3))

rf = RandomForestClassifier(n_estimators=300, random_state=RNG_SEED, n_jobs=-1).fit(Xtr, ytr)
if HAVE_XGB:
    gb = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.08, subsample=0.9,
                           eval_metric="logloss", random_state=RNG_SEED).fit(Xtr, ytr)
    gb_name = "XGBoost"
else:
    gb = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.08,
                                    random_state=RNG_SEED).fit(Xtr, ytr)
    gb_name = "GradientBoosting (xgboost fallback)"

pd.DataFrame([evaluate(rf,Xte,yte,"RandomForest"), evaluate(gb,Xte,yte,gb_name)])

### H3 — does multi-source beat single-source?

Train the same model on **task-board features only** versus **all three sources** and compare F1.

In [ ]:
single = FEATURE_GROUPS["task_board"]
rf_single = RandomForestClassifier(n_estimators=300, random_state=RNG_SEED, n_jobs=-1).fit(Xtr[single], ytr)
h3 = pd.DataFrame([
    evaluate(rf,        Xte,          yte, "Multi-source (all 3)"),
    evaluate(rf_single, Xte[single],  yte, "Single-source (task-board only)"),
])
print("H3: multi-source should have the higher F1.")
h3

## 3. Explainability Layer

**Global**: which features drive risk across all projects (SHAP mean|value|, or permutation-importance fallback).
**Local**: why *one* flagged project is at risk (SHAP values / LIME, or a z-score attribution fallback).

In [ ]:
if HAVE_SHAP:
    explainer = shap.TreeExplainer(rf)
    def _pos(v):
        if isinstance(v, list): v = v[1]
        v = np.asarray(v)
        if v.ndim == 3: v = v[..., 1]
        return v
    sv_test = _pos(explainer.shap_values(Xte))
    global_imp = pd.Series(np.abs(sv_test).mean(0), index=ALL_FEATURES).sort_values(ascending=False)
    method = "SHAP (mean |value|)"
else:
    perm = permutation_importance(rf, Xte, yte, n_repeats=8, random_state=RNG_SEED, n_jobs=-1)
    global_imp = pd.Series(perm.importances_mean, index=ALL_FEATURES).sort_values(ascending=False)
    method = "permutation importance (SHAP fallback)"

print("Global importance via", method)
plt.figure(figsize=(7,4))
global_imp.head(10)[::-1].plot(kind="barh", color="#1F3864")
plt.title(f"Global feature importance — {method}"); plt.xlabel("importance"); plt.tight_layout(); plt.show()

if HAVE_SHAP:
    try:
        shap.summary_plot(sv_test, Xte, show=True)
    except Exception as e:
        print("SHAP summary_plot skipped:", e)

In [ ]:
feat_mean, feat_std = Xtr.mean(), Xtr.std().replace(0, 1)
corr_sign = {f: (np.sign(np.corrcoef(Xtr[f], ytr)[0,1]) or 1.0) for f in ALL_FEATURES}
gimp = global_imp.reindex(ALL_FEATURES).fillna(0)

def local_drivers(instance, top=4):
    """Signed per-feature risk contribution for one project (SHAP if available, else z-score attribution)."""
    if HAVE_SHAP:
        row = _pos(explainer.shap_values(instance.to_frame().T)).reshape(-1)
        contrib = pd.Series(row, index=ALL_FEATURES)
    else:
        z = (instance[ALL_FEATURES] - feat_mean) / feat_std
        contrib = pd.Series({f: gimp[f]*z[f]*corr_sign[f] for f in ALL_FEATURES})
    contrib = contrib.sort_values(ascending=False)
    return contrib[contrib > 0].head(top)

proba_te = rf.predict_proba(Xte)[:,1]
idx  = Xte.index[np.argsort(-proba_te)[0]]        # the highest-risk project in the test set
inst = Xte.loc[idx]
print(f"Flagged project idx={idx}  P(at_risk)={rf.predict_proba(inst.to_frame().T)[0,1]:.2f}")

drivers = local_drivers(inst)
plt.figure(figsize=(7,3.4))
drivers[::-1].plot(kind="barh", color="#C0392B")
plt.title("Local risk drivers for the flagged project"); plt.tight_layout(); plt.show()

if HAVE_LIME:
    try:
        lex = lime.lime_tabular.LimeTabularExplainer(
            Xtr.values, feature_names=ALL_FEATURES, class_names=["on_track","at_risk"],
            discretize_continuous=True, random_state=RNG_SEED)
        exp = lex.explain_instance(inst.values, rf.predict_proba, num_features=6)
        print("LIME local explanation (feature -> weight):")
        for feat, w in exp.as_list():
            print(f"  {feat:40s} {w:+.3f}")
    except Exception as e:
        print("LIME skipped:", e)

drivers.round(4).to_frame("risk_contribution")

## 4. Decision Intelligence Layer

The novel piece: map each elevated risk driver to a **PMBOK-grounded recommendation** with a priority.
This turns *"why is it risky?"* into *"what do I do about it?"*.

In [ ]:
RULES = {
 "open_blockers":     (("Risk","Schedule"),        "Assign an owner to each open blocker and add a daily blocker review."),
 "overdue_ratio":     (("Schedule",),              "Re-baseline the sprint: de-scope non-critical tasks and renegotiate the date."),
 "scope_change_pct":  (("Scope",),                 "Freeze current-sprint scope; route new requests through change control."),
 "avg_cycle_time":    (("Schedule","Resource"),    "Add WIP limits and pair on long-running tasks to cut cycle time."),
 "backlog_growth":    (("Scope","Schedule"),       "Backlog outpaces throughput — reprioritize and cap intake."),
 "failed_build_ratio":(("Quality",),               "Stabilize CI: add gating tests and a green-build policy."),
 "code_churn":        (("Quality","Scope"),        "High churn signals unclear specs — clarify acceptance criteria."),
 "pr_latency":        (("Resource","Communication"),"Introduce reviewer SLAs and a review rota to cut PR wait time."),
 "commit_freq":       (("Resource",),              "Low commit velocity — check for capacity or hidden blockers."),
 "sentiment":         (("Stakeholder","Communication"),"Low team sentiment — hold a retro and address top morale blockers."),
 "comm_latency":      (("Communication",),         "Set response-time norms and a single source of truth for decisions."),
 "req_volatility":    (("Scope",),                 "Add a change-impact assessment before accepting requirement changes."),
}

def recommend(instance, top=4):
    dr = local_drivers(instance, top=top)
    mx = dr.max() if len(dr) else 1.0
    rows = []
    for f, c in dr.items():
        area, action = RULES.get(f, (("General",), "Investigate this factor with the team."))
        rows.append(dict(risk_driver=f, pmbok_area=", ".join(area), recommendation=action,
                         priority="High" if c >= 0.5*mx else "Medium", contribution=round(float(c),4)))
    return pd.DataFrame(rows)

print(f"Recommendations for flagged project idx={idx}:")
recommend(inst)

## 5. Decision-quality simulation (H2)

*Illustrative, not empirical.* For every project the model flags, we **apply its top-2 recommendations** by moving those drivers toward the "healthy-project" norm, then re-score. The share of flagged projects that become *predicted-safe* is a proxy for improved on-time delivery — the effect the Decision Intelligence Layer is designed to produce over a prediction-only tool.

In [ ]:
thr = 0.5
safe_median = Xtr[ytr == 0].median()      # healthy-project norm per feature
alpha = 0.7                                # how far we move each driver toward the norm

at_risk_idx = Xte.index[proba_te >= thr]
Xsim = Xte.loc[at_risk_idx].copy()
for i in Xsim.index:
    for f in local_drivers(Xte.loc[i], top=2).index:
        Xsim.loc[i, f] = Xsim.loc[i, f] + alpha*(safe_median[f] - Xsim.loc[i, f])

baseline_ontime = float((rf.predict_proba(Xte.loc[at_risk_idx])[:,1] < thr).mean())
xaidif_ontime   = float((rf.predict_proba(Xsim)[:,1] < thr).mean())

print(f"Flagged projects: {len(at_risk_idx)}")
print(f"Predicted-safe  | prediction-only (no action): {baseline_ontime:6.1%}")
print(f"Predicted-safe  | XAI-DIF (top-2 recs applied): {xaidif_ontime:6.1%}")
print(f"Improvement:      {xaidif_ontime - baseline_ontime:+.1%}")

plt.figure(figsize=(5,3.4))
plt.bar(["Prediction-only","XAI-DIF"], [baseline_ontime, xaidif_ontime], color=["#95A5A6","#1F3864"])
plt.ylabel("share predicted-safe"); plt.title("Decision quality (simulated, H2)"); plt.tight_layout(); plt.show()

## 6. Explanation faithfulness (supports H1)

An explanation is only useful if it is *faithful* to the model. We report two standard XAI-quality metrics:

- **Local fidelity (R²)** — how well a local linear surrogate around a prediction reproduces the model's own output. Higher = the local explanation genuinely reflects the model near that project.
- **Rank agreement (Spearman ρ)** — do two explanation methods agree on which features matter? With SHAP *and* LIME installed we compare them directly; otherwise we compare a local surrogate against the global-importance ranking.

These give you concrete numbers to cite for H1 instead of only a qualitative "the explanations look reasonable".

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr

feat_std_all = Xtr.std().replace(0, 1)

def local_surrogate(instance, n_samples=800, kernel_width=0.75, seed=0):
    """LIME-style local surrogate: perturb around the instance, weight by proximity,
    fit a weighted linear model, return (coefficients, local R^2)."""
    r = np.random.default_rng(seed)
    Z  = r.normal(0, 1, (n_samples, len(ALL_FEATURES)))
    Xp = pd.DataFrame(instance[ALL_FEATURES].values + Z*feat_std_all.values, columns=ALL_FEATURES)
    fp = rf.predict_proba(Xp)[:, 1]
    w  = np.exp(-((Z**2).sum(1)) / (2*kernel_width**2))
    lin = LinearRegression().fit(Xp, fp, sample_weight=w)
    return pd.Series(lin.coef_, index=ALL_FEATURES), lin.score(Xp, fp, sample_weight=w)

lex = None
if HAVE_LIME:
    lex = lime.lime_tabular.LimeTabularExplainer(
        Xtr.values, feature_names=ALL_FEATURES, class_names=["on_track","at_risk"],
        discretize_continuous=False, random_state=RNG_SEED)

flagged = Xte.index[np.argsort(-proba_te)[:25]]     # evaluate on the 25 highest-risk projects
fidelity, agreement = [], []
for k, i in enumerate(flagged):
    coef, r2 = local_surrogate(Xte.loc[i], seed=k); fidelity.append(r2)
    rho = np.nan
    try:
        if HAVE_SHAP and HAVE_LIME:
            shap_row = _pos(explainer.shap_values(Xte.loc[i].to_frame().T)).reshape(-1)
            e = lex.explain_instance(Xte.loc[i].values, rf.predict_proba, num_features=len(ALL_FEATURES))
            lime_vec = np.zeros(len(ALL_FEATURES))
            for j, wt in e.as_map()[1]: lime_vec[j] = wt
            rho, _ = spearmanr(np.abs(shap_row), np.abs(lime_vec))
        else:
            rho, _ = spearmanr(np.abs(coef.values), gimp.reindex(ALL_FEATURES).values)
    except Exception:
        rho, _ = spearmanr(np.abs(coef.values), gimp.reindex(ALL_FEATURES).values)
    agreement.append(rho)

lbl = "SHAP vs LIME" if (HAVE_SHAP and HAVE_LIME) else "local surrogate vs global importance"
print(f"Local fidelity R^2:                     mean={np.mean(fidelity):.3f}  "
      f"[min {np.min(fidelity):.3f}, max {np.max(fidelity):.3f}]")
print(f"Rank agreement Spearman ({lbl}): mean={np.nanmean(agreement):.3f}")
if HAVE_LIME:
    print("Tip: LIME also exposes its own local R^2 via exp.score for cross-checking fidelity.")

## 7. From PoC to thesis — plugging in real data

Everything above is model-agnostic to the *source* of the rows. To move from synthetic to real data, replace **only** `generate_project_data()` with a loader that produces the same columns:

- **Jira / Azure DevOps** → REST API or CSV export → aggregate per project/sprint into `open_blockers`, `overdue_ratio`, `avg_cycle_time`, `scope_change_pct`, `backlog_growth`.
- **GitHub** → REST/GraphQL API or the `PyGithub` library → `commit_freq`, `pr_latency`, `code_churn`, `contributors`, `failed_build_ratio`.
- **Communication** (optional) → export chat/email → run an NLP sentiment model → `sentiment`, `comm_latency`, `meeting_load`.
- **Label** → use the real outcome (delivered late / over budget) instead of the synthetic rule.
- **Benchmarks** → the PROMISE Software Engineering Repository and public software-defect datasets are good public stand-ins.

The Predictive, Explainability, and Decision Intelligence layers then run **unchanged**. Map outputs back to the paper's evaluation: the metrics table → *prediction accuracy*; SHAP/LIME + a practitioner Likert survey → *explainability effectiveness (H1)*; this simulation, later replaced by a practitioner study → *decision quality (H2)*; the multi- vs single-source table → *H3*; and add a System Usability Scale (SUS) questionnaire for the usability metric.

**Honesty note for the defense.** Present the synthetic results as a *working pipeline demonstration*, not empirical findings. The contribution a committee will value is that all three layers run end-to-end and that the explanation → recommendation mechanism produces sensible, PMBOK-grounded actions.